In [2]:
pip install -U "desi_retriever@git+https://github.com/segasai/desi_retriever" healpy astropy matplotlib


  Cloning https://github.com/segasai/desi_retriever to /private/var/folders/ss/b9pkd2ts3xqd_ryqfmgjby1r0000gn/T/pip-install-cu6iamcb/desi-retriever_b69212f3178e4d9cb3fcec5a5330b2de
  Running command git clone --filter=blob:none --quiet https://github.com/segasai/desi_retriever /private/var/folders/ss/b9pkd2ts3xqd_ryqfmgjby1r0000gn/T/pip-install-cu6iamcb/desi-retriever_b69212f3178e4d9cb3fcec5a5330b2de
  Resolved https://github.com/segasai/desi_retriever to commit c1112a5cb235fa1e5e0139bed7e7341cbab2d6f2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 23.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 49.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 57.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.3 MB/s  0:00:00
  Attempting uninstall: astropy-iers-data
    Found existing 

In [5]:
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt
import desi_retriever.dr1 as dr1

NSIDE = 64
NEST  = True

# Priority order matters!
SURVEY_PROGRAMS = [
    ("dr1", "dark"),
    ("dr1", "bright"),
    ("sv3", "dark"),
    ("sv3", "bright"),
    ("sv1", "dark"),
    ("sv1", "bright"),
]

objects = [
    (212.1152,   0.80423,    39627806405036766),
    (184.61126, -0.1305556,  39627781784471770),
    (217.36365, 52.647076,   39633290071641162),
    (245.13714, 44.380412,   39633158374688229),
]


def radec_to_hpx(ra, dec):
    theta = np.radians(90 - dec)
    phi   = np.radians(ra)
    return hp.ang2pix(NSIDE, theta, phi, nest=NEST)


for ra, dec, targetid in objects:
    hpx = radec_to_hpx(ra, dec)
    found = False

    print(f"\n🔍 TARGETID {targetid}")
    print(f"   RA={ra:.5f} DEC={dec:.5f} HPX={hpx}")

    for survey, program in SURVEY_PROGRAMS:
        try:
            spec = dr1.get_specs(
                survey=survey,
                program=program,
                hpx=hpx,
                targetid=targetid
            )[0]

            print(f"   ✅ Found spectrum in {survey}/{program}")

            wave = spec["wave"]
            flux = spec["flux"]

            plt.figure(figsize=(11, 4))
            plt.plot(wave, flux, lw=0.8)
            plt.xlabel("Wavelength [Å]")
            plt.ylabel("Flux")
            plt.title(f"DESI spectrum — {survey}/{program}\nTARGETID={targetid}")
            plt.grid(alpha=0.3)
            plt.tight_layout()
            plt.show()

            found = True
            break   # 🔑 STOP once found

        except Exception:
            continue

    if not found:
        print("   ❌ No DESI spectrum found for this object")



🔍 TARGETID 39627806405036766
   RA=212.11520 DEC=0.80423 HPX=26007
https://data.desi.lbl.gov/public/dr1//spectro/redux/iron/healpix/dr1/dark/260/26007/coadd-dr1-dark-26007.fits
https://data.desi.lbl.gov/public/dr1//spectro/redux/iron/healpix/dr1/bright/260/26007/coadd-dr1-bright-26007.fits
   ✅ Found spectrum in sv3/bright
https://data.desi.lbl.gov/public/dr1//spectro/redux/iron/healpix/sv1/dark/260/26007/coadd-sv1-dark-26007.fits
https://data.desi.lbl.gov/public/dr1//spectro/redux/iron/healpix/sv1/bright/260/26007/coadd-sv1-bright-26007.fits
   ❌ No DESI spectrum found for this object

🔍 TARGETID 39627781784471770
   RA=184.61126 DEC=-0.13056 HPX=26277
https://data.desi.lbl.gov/public/dr1//spectro/redux/iron/healpix/dr1/dark/262/26277/coadd-dr1-dark-26277.fits
https://data.desi.lbl.gov/public/dr1//spectro/redux/iron/healpix/dr1/bright/262/26277/coadd-dr1-bright-26277.fits
   ✅ Found spectrum in sv3/dark
   ✅ Found spectrum in sv3/bright
   ❌ No DESI spectrum found for this object

🔍 